# 02 — Bronze Layer (Auto Loader → VARIANT)

Uses Auto Loader (`cloudFiles`) in batch mode to read raw JSON files from the
Volume landing zone and write VARIANT-based bronze Delta tables.

Each bronze table has:
- `data VARIANT` — the full raw JSON object (semi-structured)
- `_ingestion_timestamp` — when the row was loaded
- `_source_file` — which file the row came from

Silver layer will extract typed columns using `data:field::type` syntax.

In [ ]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "ticketmaster_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Bronze schema: {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Volume path:   {VOLUME_PATH}")

In [ ]:
ENTITIES = ["events", "venues", "attractions", "classifications"]

for entity in ENTITIES:
    source_path     = f"{VOLUME_PATH}/{entity}/"
    checkpoint_path = f"{VOLUME_PATH}/_checkpoints/{entity}"
    target_table    = f"{UC_CATALOG}.{BRONZE_SCHEMA}.{entity}_raw"

    print(f"\nLoading {entity} from {source_path}")
    print(f"  Checkpoint: {checkpoint_path}")
    print(f"  Target:     {target_table}")

    # Auto Loader reads JSON files, parses each array element as a row,
    # and stores the full object as a VARIANT column.
    query = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "false")
        .option("multiLine", "true")
        .load(source_path)
        .selectExpr(
            "PARSE_JSON(TO_JSON(STRUCT(*))) AS data",
            "current_timestamp() AS _ingestion_timestamp",
            "_metadata.file_path AS _source_file"
        )
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
    )

    query.awaitTermination()
    print(f"  Done: {entity}_raw")

In [ ]:
# Show sample data and row counts
print("\n" + "=" * 60)
print("BRONZE LAYER SUMMARY")
print("=" * 60)

for entity in ENTITIES:
    table = f"{UC_CATALOG}.{BRONZE_SCHEMA}.{entity}_raw"
    count = spark.table(table).count()
    print(f"  {table}: {count:,} rows")

print("\nSample events_raw:")
spark.sql(f"""
    SELECT data:name::string AS name, data:id::string AS id,
           data:dates.start.localDate::string AS event_date,
           _ingestion_timestamp
    FROM {UC_CATALOG}.{BRONZE_SCHEMA}.events_raw
    LIMIT 5
""").show(truncate=False)